Lista 6

## Zadanie 1

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler, StandardScaler

simple_data = {
    'name': ['Piotr', 'Kuba', 'Maciek', 'Ania'],
    'income': [2500, 2300, 10000, 10200],
    'age': [21, 69, 63, 24],
}

X = pd.DataFrame(simple_data)
features = X[['income', 'age']]

# Dane oryginalne
fig = px.scatter(
    X, x='income', y='age', hover_name='name',
    title='Dane oryginalne',
    size=[20]*len(X),
    color_discrete_sequence=['cyan']
)
fig.update_layout(template='plotly_dark')
fig.show()

# KMeans bez skalowania
kmeans = KMeans(n_clusters=2, random_state=42)
labels_no_scaling = kmeans.fit_predict(features)

fig = px.scatter(
    X, x='income', y='age', color=labels_no_scaling, hover_name='name',
    title='KMeans bez skalowania',
    size=[20]*len(X),
    color_discrete_sequence=['lime', 'magenta']
)
fig.update_layout(template='plotly_dark')
fig.show()

# Skalowanie MinMax
scaler = MinMaxScaler()
scaled_minmax = scaler.fit_transform(features)

kmeans_minmax = KMeans(n_clusters=2, random_state=42)
labels_minmax = kmeans_minmax.fit_predict(scaled_minmax)

fig = px.scatter(
    x=scaled_minmax[:, 0], y=scaled_minmax[:, 1], color=labels_minmax,
    hover_name=X['name'],
    title='KMeans po skalowaniu MinMax',
    size=[20]*len(X),
    color_discrete_sequence=['orange', 'deepskyblue']
)
fig.update_layout(
    template='plotly_dark',
    xaxis_title='income (scaled)',
    yaxis_title='age (scaled)'
)
fig.show()

# Skalowanie StandardScaler
scaler = StandardScaler()
scaled_std = scaler.fit_transform(features)

kmeans_std = KMeans(n_clusters=2, random_state=42)
labels_std = kmeans_std.fit_predict(scaled_std)

fig = px.scatter(
    x=scaled_std[:, 0], y=scaled_std[:, 1], color=labels_std,
    hover_name=X['name'],
    title='KMeans po skalowaniu StandardScaler',
    size=[20]*len(X),
    color_discrete_sequence=['yellow', 'red']
)
fig.update_layout(
    template='plotly_dark',
    xaxis_title='income (std)',
    yaxis_title='age (std)'
)
fig.show()


Widzimy, że b

## Zadanie 2

In [2]:
from PIL import Image
import numpy as np
from sklearn.cluster import DBSCAN

# Pliki
IMAGES = ['figure_1_without_frame.png', 'figure_2_without_frame.png', 'figure_3_without_frame.png']
images_points = []

# Funkcja, która wyciąga punkty z obrazka za pomocą DBSCANa
def get_points(pixels):
    mask = ~( (pixels[:,:,0] > 240) & (pixels[:,:,1] > 240) & (pixels[:,:,2] > 240) )
    y, x = np.where(mask) # Najpierw zwraca y kowe wspolrzene
    coords = np.column_stack((x, y))

    db = DBSCAN(eps=2, min_samples=12).fit(coords)
    labels = db.labels_

    num_points = len(set(labels)) - (1 if -1 in labels else 0)
    print("Liczba punktów:", num_points)

    points = []

    for label in set(labels):
        if label == -1:
            continue  # szum
        cluster = coords[labels == label]
        centroid = cluster.mean(axis=0)
        points.append(centroid)

    return np.array(points)

# Dane do zadania
for image_name in IMAGES:
    with Image.open(image_name).convert('RGB') as img:
        pixels = np.array(img)
        images_points.append(get_points(pixels))

Liczba punktów: 200
Liczba punktów: 198
Liczba punktów: 195


In [3]:
from sklearn.cluster import DBSCAN, KMeans, AgglomerativeClustering
import plotly.express as px


def solve(model_name, points):
    if model_name == 'dbscan':
        model = DBSCAN(eps=4, min_samples=4).fit(points)
        labels = model.labels_
    elif model_name == 'kmeans':
        model = KMeans(n_clusters=4).fit(points)
        labels = model.labels_
    elif model_name == 'agg':
        model = AgglomerativeClustering(n_clusters=4).fit(points)
        labels = model.labels_

    fig = px.scatter(x=points[:, 0], y=points[:, 1], color=labels, title=f'Clustering: {model_name.upper()}')
    fig.update_layout(plot_bgcolor='black', paper_bgcolor='black', xaxis_showgrid=False, yaxis_showgrid=False)
    fig.update_yaxes(autorange='reversed')
    fig.show()
    
models = ['dbscan', 'kmeans', 'agg']

for model_name in models:
    for points in images_points:
        solve(model_name, points)

## Zadanie 3

In [4]:
from PIL import Image
import numpy as np
from sklearn.cluster import KMeans

with Image.open('simple_image.png').convert('RGB') as img:
    pass




## Zadanie 4

In [5]:
import numpy as np

def generate_regular_polygon(k, radius=10):
    assert k > 2
    angles = np.linspace(0, 2 * np.pi, k, endpoint=False)
    points = np.column_stack([radius * np.cos(angles),
                              radius * np.sin(angles)])
    return points

def generate_clustered_points(points, m = 10, sigma=1.0):
    all_points = []

    cov = np.array([[sigma, 0], 
                    [0, sigma]])
    
    for p in points:
        pts = np.random.multivariate_normal(p, cov, m)
        all_points.append(pts)

    return np.vstack(all_points)


from sklearn.cluster import KMeans, MiniBatchKMeans
import plotly.express as px
import plotly.graph_objects as go
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
import matplotlib.pyplot as plt
import time

K = 10
M = 30

polygon_vertices = generate_regular_polygon(K, radius=25.0)
points = generate_clustered_points(polygon_vertices, M, sigma=3)

# Grupowanie Hierarchicznie
Z = linkage(points, method="ward")

plt.figure(figsize=(14, 6))
dendrogram(Z)
plt.title("Dendrogram – Hierarchical Clustering (Ward)")
plt.xlabel("Punkty")
plt.ylabel("Dystans")
plt.show()

labels_hier = fcluster(Z, t=K, criterion="maxclust")

fig = px.scatter(
    x=points[:, 0],
    y=points[:, 1],
    color=labels_hier.astype(str),
    title=f"Hierarchical clustering – {K} clusters"
)

fig.update_yaxes(scaleanchor="x", scaleratio=1)
fig.update_layout(width=800, height=800)
fig.show()


# Podpunkt 2
for k in range(2, K + 1):
    kms = KMeans(n_clusters=k, random_state=67)
    labels = kms.fit_predict(points)

    fig = px.scatter(
        x=points[:, 0],
        y=points[:, 1],
        color=labels.astype(str),
        title=f"KMeans: {k} grup"
    )

    fig.add_trace(
        go.Scatter(
            x=polygon_vertices[:, 0],
            y=polygon_vertices[:, 1],
            mode="markers",
            marker=dict(
                symbol="x",
                size=12,
                color="black"
            ),
            name="wierzchołki"
        )
    )

    fig.update_yaxes(scaleanchor="x", scaleratio=1)
    fig.update_xaxes(constrain="domain")
    fig.update_yaxes(constrain="domain")
    fig.update_layout(width=800, height=800)
    fig.show()

# Sprawdzanie prędkości
K = 1000
M = 10000

polygon_vertices = generate_regular_polygon(K, radius=25.0)
points = generate_clustered_points(polygon_vertices, M, sigma=3)

# --- KMeans ---
t0 = time.time()
kmeans = KMeans(n_clusters=K, random_state=123)
labels_kmeans = kmeans.fit_predict(points)
t_kmeans = time.time() - t0

# --- MiniBatchKMeans ---
t0 = time.time()
mbk = MiniBatchKMeans(n_clusters=K, batch_size=2000, random_state=123)
labels_mbk = mbk.fit_predict(points)
t_mbk = time.time() - t0

print(f"Czas KMeans: {t_kmeans:.02f}s")
print(f"Czas MiniBatchKMeans: {t_mbk:.02f}s")
